# YOLO 2회차 추가 실습 — 말로 찾는 탐지기 (YOLOE)

실습 A에서는 상자를 찾기 위해 라벨이 달린 이미지 1,920장으로 모델을 학습했습니다. 그런데 **학습 없이, 찾고 싶은 물체의 이름만 말해서** 찾을 수는 없을까요?
YOLOE는 글자(텍스트 프롬프트)로 찾을 대상을 지정하는 **오픈 보캐뷸러리(open-vocabulary) 탐지 모델**입니다. 이 실습에서는 YOLOE를 써 보고, 어떤 단어로 부르느냐에 따라 성능이 얼마나 달라지는지 **프롬프트 대결**을 하고, 직접 학습한 모델과도 맞붙여 봅니다.

| 단계 | 만드는 것 | 결과물 |
|---|---|---|
| Step 1 | 학습 없이 말로 탐지하기 | 탐지 결과 이미지 |
| Step 2 | 같은 사진, 다른 단어 — 프롬프트에 따라 달라지는 결과 | 비교 그림 |
| Step 3 | 프롬프트 대결 — 단어별 mAP 측정 | `prompt_battle.csv` |
| Step 4 | 직접 학습한 모델과 대결 — 정확도·속도·준비 비용 | 비교 표 |
| Step 5 | 응용: 안전모 미착용자 찾기 + 정답 라벨로 채점 | `helmet_check.csv` |

**데이터셋** (Ultralytics 공개 데이터셋, 자동 다운로드)
- *Package Segmentation* — 실습 A와 같은 택배 상자 CCTV 이미지 (검증 188장)
- *Construction-PPE* — 건설 현장 작업자 안전장비 이미지 (학습 1,132장 · 검증 143장 · 테스트 141장, 안전모·조끼·장갑 등 11개 클래스)

**실행 환경**: Google Colab, 런타임 유형 **T4 GPU**

## 0. 준비

In [ ]:
!pip -q install ultralytics
!pip -q install git+https://github.com/ultralytics/CLIP.git

In [ ]:
import os, json, time, shutil, glob
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from ultralytics import YOLO, YOLOE
from ultralytics.data.utils import check_det_dataset

print('GPU 사용 가능:', torch.cuda.is_available())

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_DRIVE = True
if IN_COLAB and USE_DRIVE:
    drive.mount('/content/drive')
    DAY2_DIR = Path('/content/drive/MyDrive/yolo_day2')
else:
    DAY2_DIR = Path('outputs')
SAVE_DIR = DAY2_DIR / 'prompt'
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print('결과 저장 위치:', SAVE_DIR.resolve())

def show_bgr(img, title=None, size=(9, 6)):
    plt.figure(figsize=size)
    plt.imshow(img[..., ::-1]); plt.axis('off')
    if title:
        plt.title(title)
    plt.show()

def show_grid(imgs, titles=None, cols=3, size=4):
    rows = (len(imgs) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * rows))
    for ax, i in zip(np.array(axes).ravel(), range(rows * cols)):
        ax.axis('off')
        if i < len(imgs):
            ax.imshow(imgs[i][..., ::-1])
            if titles:
                ax.set_title(titles[i], fontsize=9)
    plt.tight_layout(); plt.show()

def save_out(path):
    dst = SAVE_DIR / Path(path).name
    if Path(path).resolve() != dst.resolve():
        shutil.copy(path, dst)
    return dst

In [ ]:
# 데이터셋 두 개 내려받기 (처음 한 번, 합계 약 280MB)
PKG = check_det_dataset('package-seg.yaml')
PPE = check_det_dataset('construction-ppe.yaml')
PKG_ROOT, PPE_ROOT = Path(PKG['path']), Path(PPE['path'])
print('package-seg     :', PKG_ROOT, PKG['names'])
print('construction-ppe:', PPE_ROOT, PPE['names'])

---
## Step 1. 학습 없이 말로 탐지하기

### YOLOE는 어떻게 단어를 알아듣나

```
"cardboard box" ──▶ 텍스트 인코더(MobileCLIP) ──▶ 단어 벡터 ─┐
                                                              ├─▶ 비슷한 영역 = 탐지 결과
이미지 ───────────▶ YOLO 백본 ─────────────────▶ 영역 벡터 ──┘
```

- 텍스트 인코더가 단어를 숫자 벡터로 바꾸고, 이미지의 각 영역 벡터와 **얼마나 비슷한지** 비교합니다.
- 그래서 학습 때 클래스 목록에 없던 단어도 찾을 수 있습니다. 대신 단어를 고르는 방식에 따라 결과가 크게 달라집니다.
- `set_classes([...])`로 찾을 단어 목록을 정합니다. 처음 호출할 때 텍스트 인코더 가중치를 내려받아 시간이 조금 걸립니다.

In [ ]:
model = YOLOE('yoloe-26s-seg.pt')
model.set_classes(['person', 'helmet', 'safety vest', 'cardboard box'])

ppe_test = sorted((PPE_ROOT / 'images' / 'test').glob('*'))
pkg_val = sorted((PKG_ROOT / 'images' / 'val').glob('*.jpg'))
samples = [ppe_test[0], ppe_test[2], ppe_test[5], pkg_val[3], pkg_val[95], pkg_val[150]]

imgs, titles = [], []
for p in samples:
    r = model.predict(str(p), conf=0.25, verbose=False)[0]
    imgs.append(r.plot())
    names = [r.names[int(c)] for c in r.boxes.cls]
    titles.append(', '.join(f'{n} {names.count(n)}' for n in dict.fromkeys(names)) or 'nothing found')
show_grid(imgs, titles)

**확인해 보기** — 공사 현장 사진에서는 사람·안전모·조끼를 잘 찾지만, 택배 분류장 CCTV에서는 상자를 거의 찾지 못합니다. 왜 그럴까요?
YOLOE가 배운 인터넷 사진 속 '상자'는 정면에서 또렷하게 찍힌 모습이 많습니다. 위에서 내려다보는 흐릿한 CCTV 화면, 붉은 조명 아래의 상자는 낯선 모습입니다.

---
## Step 2. 같은 사진, 다른 단어

같은 CCTV 사진에 단어만 바꿔 넣고 결과를 비교합니다. 사람에게는 'package'와 'cardboard box'가 같은 물건이지만 모델에게는 다른 벡터입니다.

In [ ]:
PROMPTS = ['package', 'box', 'cardboard box', 'parcel', 'carton box']
probe = pkg_val[60]
LOW_CONF = 0.003          # CCTV 상자에는 점수가 아주 낮게 나오므로 기준을 크게 낮춰서 봅니다

imgs, titles = [], []
for word in PROMPTS:
    model.set_classes([word])
    r = model.predict(str(probe), conf=LOW_CONF, verbose=False)[0]
    conf = r.boxes.conf.cpu().numpy()
    imgs.append(r.plot(labels=False))
    titles.append(f'"{word}": {len(conf)} boxes, max conf {conf.max():.3f}' if len(conf) else f'"{word}": 0 boxes')
gt_n = len(open(str(probe).replace('images', 'labels').rsplit('.', 1)[0] + '.txt').read().strip().splitlines())
show_grid(imgs, titles)
print('이 사진의 정답 상자 수:', gt_n)

두 가지를 확인하세요.
- 단어마다 찾은 상자 수가 다릅니다. 'package'는 거의 못 찾고 'cardboard box'나 'parcel'은 여러 개를 찾습니다.
- **점수(conf)가 0.1 안팎으로 매우 낮습니다.** 낯선 장면에서 제로샷 모델은 정답을 맞혀도 자신 없는 점수를 줍니다. 그래서 기본 기준(0.25)으로는 아무것도 안 보이고, 기준을 아주 낮춰야 보입니다. 실제로 쓰려면 이 기준도 데이터로 정해야 합니다.

한 장만 보고 단어를 고르면 우연에 휘둘립니다. Step 3에서는 **검증 이미지 188장 전체**로 단어별 성적을 매깁니다.

---
## Step 3. 프롬프트 대결 — 단어별 mAP

YOLOE의 `val()`은 데이터셋 설정 파일의 **클래스 이름을 그대로 프롬프트로** 씁니다. 그래서 이름만 바꾼 설정 파일을 단어마다 만들고 평가하면 됩니다.
이미지와 정답 라벨은 같고 **단어만 다르므로**, 점수 차이는 순전히 단어 선택 때문입니다.

In [ ]:
def make_yaml(word, path):
    cfg = {'path': str(PKG_ROOT), 'train': 'images/train', 'val': 'images/val', 'names': {0: word}}
    Path(path).write_text(json.dumps(cfg))     # JSON은 YAML의 부분집합이라 그대로 읽힘
    return path

battle = []
for word in PROMPTS:
    yml = make_yaml(word, f'pkg_{word.replace(" ", "_")}.yaml')
    m = YOLOE('yoloe-26s-seg.pt')
    res = m.val(data=yml, split='val', imgsz=640, batch=16, verbose=False, plots=False)
    battle.append({'prompt': word, 'precision': round(res.box.mp, 3), 'recall': round(res.box.mr, 3),
                   'map50': round(res.box.map50, 3), 'map50_95': round(res.box.map, 3)})
    print(f'{word:15s} mAP50 {res.box.map50:.3f}')

battle = pd.DataFrame(battle).sort_values('map50', ascending=False).reset_index(drop=True)
battle.to_csv('prompt_battle.csv', index=False); save_out('prompt_battle.csv')
battle

In [ ]:
plt.figure(figsize=(8, 3.5))
bars = plt.bar(battle['prompt'], battle['map50'], color=['tab:green'] + ['tab:gray'] * (len(battle) - 1))
for b, v in zip(bars, battle['map50']):
    plt.text(b.get_x() + b.get_width() / 2, v + 0.01, f'{v:.2f}', ha='center')
plt.ylim(0, 1); plt.ylabel('mAP50'); plt.title('Prompt battle on package-seg val (188 images)')
plt.show()
BEST_PROMPT = battle.loc[0, 'prompt']
print('우승 단어:', BEST_PROMPT)

> mAP는 점수의 **순서**(정답 박스가 헛박스보다 높은 점수를 받는가)로 계산하므로, 점수가 0.01처럼 낮아도 순서만 잘 맞으면 높게 나옵니다. 그래서 mAP가 괜찮아도 실제 서비스에서는 Step 2에서 본 것처럼 낮은 기준을 따로 정해야 합니다.

**확인해 보기**
- 1등 단어와 꼴찌 단어의 mAP50 차이는 얼마인가요? 모델을 바꾸지 않고 **단어 하나로** 생긴 차이입니다.
- 정밀도(precision)와 재현율(recall)을 보세요. 어떤 단어는 많이 찾지만 헛탐지가 많고, 어떤 단어는 적게 찾지만 정확합니다.
- 나만의 단어를 `PROMPTS`에 추가해 순위를 바꿔 보세요. 예: `'brown box'`, `'shipping box'`, `'box on conveyor'`

---
## Step 4. 직접 학습한 모델과 대결

실습 A에서 학습해 Drive에 저장한 `box_seg_best.pt`가 있으면 불러와 **같은 검증 이미지**로 평가합니다. 정확도뿐 아니라 **이미지 한 장 처리 시간**과 **준비 비용**(라벨 이미지 수, 학습 시간)도 함께 비교합니다.

In [ ]:
def ms_per_image(m, images, n=40, **kw):
    m.predict(str(images[0]), verbose=False, **kw)              # 예열 (첫 호출은 느림)
    t0 = time.time()
    for p in images[:n]:
        m.predict(str(p), verbose=False, **kw)
    return (time.time() - t0) / n * 1000

compare = []
zs = YOLOE('yoloe-26s-seg.pt'); zs.set_classes([BEST_PROMPT])
compare.append({'model': f'YOLOE zero-shot ("{BEST_PROMPT}")', 'labeled_images': 0, 'training': 'none',
                'map50': battle.loc[0, 'map50'], 'ms_per_image': round(ms_per_image(zs, pkg_val), 1)})

LAB_A_MODEL = DAY2_DIR / 'box_seg_best.pt'
if LAB_A_MODEL.exists():
    ft = YOLO(LAB_A_MODEL)
    res = ft.val(data='package-seg.yaml', split='val', batch=16, verbose=False, plots=False)
    compare.append({'model': 'Lab A fine-tuned (yolo26n-seg)', 'labeled_images': 1920, 'training': 'yes',
                    'map50': round(res.box.map50, 3), 'ms_per_image': round(ms_per_image(ft, pkg_val), 1)})
else:
    print('실습 A 모델이 없습니다. 실습 A를 먼저 실행하면 이 표에 함께 나옵니다.')

compare = pd.DataFrame(compare)
compare.to_csv('model_compare.csv', index=False); save_out('model_compare.csv')
compare

**생각해 보기 — 언제 무엇을 쓸까**

| 상황 | 알맞은 선택 |
|---|---|
| 새 물체를 **오늘 당장** 찾아봐야 함, 라벨 없음 | YOLOE 제로샷 + 좋은 단어 찾기 |
| 정확도가 서비스 품질을 좌우함 (카운팅·과금·안전) | 라벨을 모아 직접 학습 |
| 라벨링 비용을 줄이고 싶음 | YOLOE로 **라벨 초안**을 만들고 사람이 고쳐 학습 → *오토 라벨링 공장* 실습 |

YOLOE의 `s` 모델은 학습한 `n` 모델보다 크기 때문에 느릴 수 있습니다. 엣지 장비(3회차 Jetson)에서는 속도 차이가 더 중요해집니다.

---
## Step 5. 응용 — 안전모 미착용자 찾기

YOLOE로 `person`과 `helmet`을 찾은 뒤, **규칙**을 붙여 '안전모를 쓰지 않은 사람'을 찾습니다.

```
사람 박스의 위쪽 35% (머리 영역) 안에 안전모 박스의 중심이 있는가?
   예  → 착용 (초록)
 아니요 → 미착용 (빨강)
```

Construction-PPE 데이터셋에는 사람이 직접 단 `no_helmet`(안전모 미착용) 라벨이 있습니다. 이 라벨로 우리 규칙을 **채점**합니다.

In [ ]:
HEAD_RATIO = 0.35

def helmet_check(r):
    """YOLOE 결과 → [(사람 박스, 착용 여부)]"""
    boxes, cls = r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int)
    names = r.names
    persons = boxes[[names[c] == 'person' for c in cls]]
    helmets = boxes[[names[c] == 'helmet' for c in cls]]
    out = []
    for x1, y1, x2, y2 in persons:
        head_y2 = y1 + (y2 - y1) * HEAD_RATIO
        worn = any(x1 <= (hx1 + hx2) / 2 <= x2 and y1 - (y2 - y1) * 0.1 <= (hy1 + hy2) / 2 <= head_y2
                   for hx1, hy1, hx2, hy2 in helmets)
        out.append(((x1, y1, x2, y2), worn))
    return out

def draw_check(img, checks):
    for (x1, y1, x2, y2), worn in checks:
        col = (0, 200, 0) if worn else (0, 0, 255)
        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), col, 3)
        label = 'HELMET OK' if worn else 'NO HELMET'
        cv2.rectangle(img, (int(x1), int(y1) - 24), (int(x1) + 130, int(y1)), col, -1)
        cv2.putText(img, label, (int(x1) + 4, int(y1) - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    return img

model.set_classes(['person', 'helmet'])
imgs, titles = [], []
for p in ppe_test[:6]:
    r = model.predict(str(p), conf=0.3, verbose=False)[0]
    ch = helmet_check(r)
    imgs.append(draw_check(r.orig_img.copy(), ch))
    titles.append(f'{sum(not w for _, w in ch)} without helmet / {len(ch)} people')
show_grid(imgs, titles)

### 정답 라벨로 채점하기

이미지 단위로 채점합니다. 정답 라벨에 `no_helmet`이 하나라도 있으면 '미착용자 있음(양성)'입니다.

| | 규칙: 미착용자 있음 | 규칙: 없음 |
|---|---|---|
| **정답: 있음** | TP (잘 찾음) | FN (놓침 — 위험) |
| **정답: 없음** | FP (헛경보) | TN |

In [ ]:
NO_HELMET = [k for k, v in PPE['names'].items() if v == 'no_helmet'][0]

rows = []
for p in ppe_test:
    lbl = PPE_ROOT / 'labels' / 'test' / (p.stem + '.txt')
    gt_cls = [int(l.split()[0]) for l in lbl.read_text().strip().splitlines()] if lbl.exists() else []
    r = model.predict(str(p), conf=0.3, verbose=False)[0]
    ch = helmet_check(r)
    rows.append({'image': p.name, 'gt_no_helmet': int(NO_HELMET in gt_cls),
                 'pred_no_helmet': int(any(not w for _, w in ch)), 'people_found': len(ch)})
check = pd.DataFrame(rows)
check.to_csv('helmet_check.csv', index=False); save_out('helmet_check.csv')

tp = int(((check.gt_no_helmet == 1) & (check.pred_no_helmet == 1)).sum())
fn = int(((check.gt_no_helmet == 1) & (check.pred_no_helmet == 0)).sum())
fp = int(((check.gt_no_helmet == 0) & (check.pred_no_helmet == 1)).sum())
tn = int(((check.gt_no_helmet == 0) & (check.pred_no_helmet == 0)).sum())
print(f'테스트 이미지 {len(check)}장 | TP {tp}  FN {fn}  FP {fp}  TN {tn}')
print(f'재현율(미착용 이미지를 찾은 비율) {tp / max(tp + fn, 1):.2f} | 정밀도(경보가 맞은 비율) {tp / max(tp + fp, 1):.2f}')

In [ ]:
# 놓친 사례(FN)와 헛경보(FP) 눈으로 보기
imgs, titles = [], []
for kind, cond in [('FN (missed)', (check.gt_no_helmet == 1) & (check.pred_no_helmet == 0)),
                   ('FP (false alarm)', (check.gt_no_helmet == 0) & (check.pred_no_helmet == 1))]:
    for name in check.loc[cond, 'image'].head(3):
        r = model.predict(str(PPE_ROOT / 'images' / 'test' / name), conf=0.3, verbose=False)[0]
        imgs.append(draw_check(r.orig_img.copy(), helmet_check(r))); titles.append(f'{kind}: {name}')
if imgs:
    show_grid(imgs, titles)

**확인해 보기**
- 놓친 사례(FN)는 왜 생겼나요? 사람이 아예 안 잡혔는지, 모자를 안전모로 착각했는지 구분해 보세요.
- `HEAD_RATIO`, `conf`, 프롬프트(`'helmet'` → `'hard hat'`, `'safety helmet'`)를 바꿔 재현율을 올려 보세요. 안전 시스템에서는 헛경보보다 **놓침이 더 위험**하다는 점을 기억하세요.

결과 파일 (`SAVE_DIR`에 저장)

| 파일 | 내용 |
|---|---|
| `prompt_battle.csv` | 단어별 성적 (`prompt, precision, recall, map50, map50_95`) |
| `model_compare.csv` | 제로샷 vs 학습 모델 (`model, labeled_images, training, map50, ms_per_image`) |
| `helmet_check.csv` | 이미지별 채점 (`image, gt_no_helmet, pred_no_helmet, people_found`) |

---
## 도전 과제

1. **문장형 프롬프트** — `'a person not wearing a helmet'`처럼 문장으로 직접 찾으면 규칙 방식보다 나을까요? Step 5 채점 코드로 비교해 보세요.
2. **조끼까지** — `safety vest`를 추가해 '안전모와 조끼를 모두 착용했는지' 판정하도록 `helmet_check()`를 확장하세요. 데이터셋에는 조끼 미착용 라벨이 없으니 채점 기준을 어떻게 세울지도 생각해 보세요.
3. **보이는 예시로 찾기** — YOLOE는 단어 대신 **예시 박스(visual prompt)** 로도 찾을 수 있습니다. 상자 하나에 박스를 그려 주고 나머지를 찾게 해 보세요. (Ultralytics 문서의 `YOLOEVPSegPredictor` 예제 참고)
4. **다른 데이터로** — 캐글이나 공공데이터에서 받은 사진에 원하는 단어를 넣어 보세요. 어떤 물체는 잘 되고 어떤 물체는 안 되는지 표로 정리합니다.

## 정리

- YOLOE는 **단어로 찾을 대상을 정하는** 탐지 모델입니다. 학습 없이 바로 쓸 수 있습니다.
- 같은 모델이라도 **어떤 단어로 부르느냐**에 따라 성능이 크게 달라집니다. 단어도 데이터로 고릅니다.
- 정확도가 중요한 서비스에서는 아직 **직접 학습한 모델**이 앞섭니다. 제로샷은 빠른 시작과 라벨 초안 만들기에 강합니다.
- 탐지 결과에 **간단한 규칙**을 더하면 '안전모 미착용' 같은 현장 질문에 답할 수 있고, 정답 라벨로 그 규칙을 채점해야 믿고 쓸 수 있습니다.